In [1]:
import sys
sys.path.append(fr"S:/ML_simulation_uncertainty_esk/Modelling")

In [2]:
# Import value change packages
from valueChange import value_change

# Packages for spatial data manipulation
from shapely.geometry import Polygon, LineString            # For creating polygons to change raster values
from shapely.ops import linemerge, unary_union, polygonize  # For spliting polygons using linestring
import geopandas as gpd                                     # For manipulating shape files
import rioxarray as rxr                                     # For manipulating spatial data under xarray array format

# Packages for data manipulation
import numpy as np
import pandas as pd
import json

In [3]:
main_dir = r"S:/buller_remove_injection_points_002"

In [4]:
# Get coordinates of area to remove
area_to_remove = [
    [1489805, 5367645],
    [1489805, 5367675],
    [1489825, 5367675],
    [1489825, 5367645],
    [1489805, 5367645]
]

# Create polygon
area_polygon = Polygon(area_to_remove)

# Convert into geopandas dataframe
polygon_dataframe = gpd.GeoDataFrame(geometry=[area_polygon], crs=2193)

# Write out shapefile
polygon_dataframe.to_file(
    fr"{main_dir}/area_to_remove.shp", crs=2193
)

In [5]:
# Convert netcdf into tiff
buller_nc = rxr.open_rasterio(fr"{main_dir}/buller_sd.nc")
buller_nc.rio.to_raster(fr"{main_dir}/buller_sd.tiff")

In [6]:
shape = fr"{main_dir}/area_to_remove.shp"
tiff = fr"{main_dir}/buller_sd.tiff"

value_change(shape, tiff, 0, True)

In [7]:
# Convert back to netcdf
buller_tiff = rxr.open_rasterio(fr"{main_dir}/buller_sd.tiff")
buller_tiff.rio.to_raster(fr"{main_dir}/buller_removed_sd.nc")

In [8]:
# Read new sd
buller_removed_sd = rxr.open_rasterio(fr"{main_dir}/buller_removed_sd.nc")

# Replace -9999 with nan
buller_nan_sd = buller_removed_sd.where(buller_removed_sd.values != 0, np.nan)

In [9]:
# Fill nan
buller_filled_sd = buller_nan_sd.interpolate_na(dim='x', method='zero')

In [10]:
buller_filled_sd.rio.to_raster(fr"{main_dir}/buller_filtered_sd.tiff")

In [11]:
# Get coordinates of area to remove
area_to_remove2 = [
    [1489825, 5367655],
    [1489825, 5367665],
    [1489835, 5367665],
    [1489835, 5367655],
    [1489825, 5367655]
]

# Create polygon
area_polygon2 = Polygon(area_to_remove2)

# Convert into geopandas dataframe
polygon_dataframe2 = gpd.GeoDataFrame(geometry=[area_polygon2], crs=2193)

# Write out shapefile
polygon_dataframe2.to_file(
    fr"{main_dir}/area_to_remove2.shp", crs=2193
)

In [12]:
shape = fr"{main_dir}/area_to_remove2.shp"
tiff = fr"{main_dir}/buller_filtered_sd.tiff"

value_change(shape, tiff, 0, True)

In [13]:
# Convert back to netcdf
buller_tiff2 = rxr.open_rasterio(fr"{main_dir}/buller_filtered_sd.tiff")
buller_tiff2.rio.to_raster(fr"{main_dir}/buller_filtered_sd2.nc")

In [14]:
# Read new sd
buller_removed_sd2 = rxr.open_rasterio(fr"{main_dir}/buller_filtered_sd2.nc")

# Replace -9999 with nan
buller_nan_sd2 = buller_removed_sd2.where(buller_removed_sd2.values != 0, np.nan)

In [15]:
# Fill nan
buller_filled_sd2 = buller_nan_sd2.interpolate_na(dim='x', method='nearest')

In [16]:
buller_filled_sd2.rio.to_raster(fr"{main_dir}/buller_filtered_sd3.nc")